# 08 Roboflowデータセットでopen_palm検出を転移学習

RoboflowからYOLOv8形式のデータセットを取得し、`yolov8n.pt` を初期重みとして、開いた手のひら `open_palm` だけを検出するモデルを学習します。

In [ ]:
!pip -q install ultralytics roboflow

In [ ]:
from pathlib import Path
import sys

try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT_PATH = Path("/content/drive/MyDrive/cnn-hands-on")
except Exception:
    ROOT_PATH = Path.cwd()

if str(ROOT_PATH) not in sys.path:
    sys.path.append(str(ROOT_PATH))

print("ROOT_PATH:", ROOT_PATH)

## Roboflowからデータセットを取得する

`YOUR_API_KEY`、`YOUR_WORKSPACE`、`YOUR_PROJECT`、`VERSION_NUMBER` を自分のRoboflowプロジェクトに合わせて変更します。

In [ ]:
from roboflow import Roboflow

ROBOFLOW_API_KEY = "YOUR_API_KEY"
ROBOFLOW_WORKSPACE = "YOUR_WORKSPACE"
ROBOFLOW_PROJECT = "YOUR_PROJECT"
VERSION_NUMBER = 1

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
version = project.version(VERSION_NUMBER)
dataset = version.download("yolov8")

DATA_YAML = Path(dataset.location) / "data.yaml"
print("dataset:", dataset.location)
print("data.yaml:", DATA_YAML)

## yolov8n.ptを初期重みにして学習する

まずは軽い `yolov8n.pt` で流れを確認します。ColabのGPUを有効にしてから実行してください。

In [ ]:
import torch
from ultralytics import YOLO

print("CUDA available:", torch.cuda.is_available())

model = YOLO("yolov8n.pt")

results = model.train(
    data=str(DATA_YAML),
    epochs=30,
    imgsz=640,
    batch=16,
    patience=10,
    project="runs/open_palm",
    name="yolov8n_transfer",
)

## 学習結果を確認する

In [ ]:
from IPython.display import Image, display

run_dir = Path("runs/open_palm/yolov8n_transfer")
best_model_path = run_dir / "weights" / "best.pt"

print("best model:", best_model_path)

for image_name in ["results.png", "confusion_matrix.png"]:
    image_path = run_dir / image_name
    if image_path.exists():
        display(Image(filename=str(image_path)))
    else:
        print(f"{image_path} はまだ作成されていません")

In [ ]:
best_model = YOLO(str(best_model_path))
metrics = best_model.val(data=str(DATA_YAML), imgsz=640)
metrics

## テスト画像で推論する

In [ ]:
test_images = sorted((Path(dataset.location) / "test" / "images").glob("*"))

if test_images:
    sample_images = test_images[:5]
    predict_results = best_model.predict(
        source=[str(path) for path in sample_images],
        conf=0.25,
        save=True,
    )
    save_dir = Path(predict_results[0].save_dir)
    for path in sample_images:
        output_path = save_dir / path.name
        if output_path.exists():
            display(Image(filename=str(output_path)))
else:
    print("test/images が見つかりませんでした")

## Webカメラで撮影してopen_palmを検出する

In [ ]:
from utils.camera import take_photo

photo_path = take_photo("open_palm_camera.jpg")
display(Image(filename=photo_path))

In [ ]:
camera_results = best_model.predict(
    source="open_palm_camera.jpg",
    conf=0.25,
    save=True,
)
output_path = Path(camera_results[0].save_dir) / "open_palm_camera.jpg"
display(Image(filename=str(output_path)))